In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
def extraccion_datos(var, pos, ruta_datos):
    dato = []
    tiempo = []
    with open(ruta_datos, "r", encoding="utf-8") as file:
        for line in file:
            if var in line:
                campos = line.strip().split()
                if campos[2] == var:
                    dato.append(float(campos[pos]))
                    tiempo.append(float(campos[0]))
    return np.array(dato), np.array(tiempo)

In [3]:
def llevar_a_t_comun(t_comun, t_var, var, check=False):
    var_interp = []
    t_var_interp = []

    i = 0  # índice en t_var
    last_value = var[0]
    last_time = t_var[0]

    for t in t_comun:
        while i < len(t_var) and t_var[i] <= t:
            last_value = var[i]
            last_time = t_var[i]
            i += 1
        var_interp.append(last_value)
        t_var_interp.append(last_time)

    return np.array(var_interp), np.array(t_var_interp)

In [4]:
from datetime import datetime, timedelta
def gps_to_datetime_local(week, tow_ms):
    gps_epoch = datetime(1980, 1, 6)
    # GPS time
    gps_time = gps_epoch + timedelta(weeks=int(week), milliseconds=int(tow_ms))
    # Ajuste a UTC (restando los 19 segundos de diferencia GPS–UTC)
    utc_time = gps_time - timedelta(seconds=19)
    # Ajuste a hora local española (CEST: UTC+2 en verano)
    local_time = utc_time
    return local_time

In [5]:
def hhmmss_a_segundos(t_str):
    try:
        h, m, s = map(int, t_str.strip().split(":"))
        return h * 3600 + m * 60 + s
    except:
        return None

In [6]:
#Definimos la ruta donde están los datos de la sonda y la de salida del csv fixeado
nombre_archivo = "datos_sensor.csv"
home_dir = os.path.expanduser("~")

ruta_entrada = os.path.join(home_dir, "PprzGCS", "Planificacion", "Resources", "Medidas_sonda", nombre_archivo)
ruta_salida = os.path.join(home_dir, "PprzGCS", "Planificacion", "Medidas_sonda", nombre_archivo)

In [7]:
nombre_archivo_mision = "25_07_16__09_18_40.data"  # nombre que recibes desde Qt
ruta_datos_mision = os.path.join(home_dir, "paparazzi", "var", "logs", nombre_archivo_mision)

In [8]:
nombre_datos_sonda = "piscinaUCM16-07-25_250716.csv"
ruta_datos_sonda = os.path.join(home_dir, "Desktop", nombre_datos_sonda)

In [9]:
#Mensajes de los que extraer datos
profundidades_pprz, t_prof_pprz = extraccion_datos("SERIAL_COM", 15, ruta_datos_mision) #Da profundidad de la sonda del log
mode, t_mode = extraccion_datos("SERIAL_COM", 3, ruta_datos_mision) #Si está en 2 -> Manual en 4->Auto
static_control, t_static_control = extraccion_datos("STATIC_CONTROL", 3, ruta_datos_mision) #Si está en static_control


N_tramos_5s = int(t_prof_pprz[-1]/5)
#Vamos a crear un vector de tiempos de 5s en 5s
tiempos_5s = np.linspace(0, t_prof_pprz[-1], N_tramos_5s)
#Estos son para sacar tiempo absoluto de paparazzi
week, t_week = extraccion_datos("GPS", 10, ruta_datos_mision) #Semana desde 6 de Enero de 1980
tow, t_tow = extraccion_datos("GPS", 11, ruta_datos_mision) #tow = time on week
utm_zone, t_utm_zone = extraccion_datos("GPS", 12, ruta_datos_mision) #tow = time on week


static_control, t_static_control = llevar_a_t_comun(tiempos_5s, t_static_control, static_control)
mode, t_mode = llevar_a_t_comun(tiempos_5s, t_mode, mode)
mode, t_mode = llevar_a_t_comun(tiempos_5s, t_mode, mode)

week, t_week = llevar_a_t_comun(tiempos_5s, t_week, week)
tow, t_tow = llevar_a_t_comun(tiempos_5s, t_tow, tow)
profundidades_pprz, t_prof_pprz = llevar_a_t_comun(tiempos_5s, t_prof_pprz, profundidades_pprz)

#Para calular el tiempo absoluto a partir de los mensajes
# Vectoriza para usar con arrays
vectorized_gps_to_datetime_local = np.vectorize(gps_to_datetime_local)

# Aplica a tus arrays
result_local_time = vectorized_gps_to_datetime_local(week, tow)

In [10]:
df_sonda = pd.read_csv(ruta_datos_sonda, sep=";")
#print(df_sonda.columns.tolist())

#print(df_sonda)
date_sonda = []
tiempo_sonda = []
profundidad_sonda = []
# temperatura = []
# ph = []
# do_sat = []
# do = []
# blue = []
# chl = []
df_sonda["Columna1"] = df_sonda["Columna1"].apply(hhmmss_a_segundos)

for i in range(len(df_sonda)):
    date_sonda.append(df_sonda.iloc[i]["Date (ISO8601)"])
    tiempo_sonda.append(df_sonda.iloc[i]["Columna1"])
    profundidad_sonda.append(df_sonda.iloc[i]["Depth (m)"])
    # temperatura.append(df_rasp.iloc[i+1]["Temperatura"])
    # ph.append(df_rasp.iloc[i+1]["pH"])
    # do_sat.append(df_rasp.iloc[i+1]["DO_SAT"])
    # do.append(df_rasp.iloc[i+1]["DO"])
    # blue.append(df_rasp.iloc[i+1]["Blue"])
    # chl.append(df_rasp.iloc[i+1]["Chl"])

In [11]:
df_rasp = pd.read_csv(ruta_entrada)
#print(df_sonda)
tiempo = []
profundidad = []
temperatura = []
ph = []
do_sat = []
do = []
blue = []
chl = []
df_rasp = df_rasp[df_rasp["Tiempo"] != "Tiempo"].copy()
for i in range(0, len(df_rasp)-2, 2):
    if df_rasp.iloc[i]["Tiempo"] != "Tiempo" and df_rasp.iloc[i+1]["Profundidad"] != "Profundidad":
        tiempo.append(df_rasp.iloc[i]["Tiempo"])
        profundidad.append(df_rasp.iloc[i+1]["Profundidad"])
        temperatura.append(df_rasp.iloc[i+1]["Temperatura"])
        ph.append(df_rasp.iloc[i+1]["pH"])
        do_sat.append(df_rasp.iloc[i+1]["DO_SAT"])
        do.append(df_rasp.iloc[i+1]["DO"])
        blue.append(df_rasp.iloc[i+1]["Blue"])
        chl.append(df_rasp.iloc[i+1]["Chl"])

tiempo_fixed = pd.to_datetime(tiempo, format="%a %b %d %H:%M:%S %Y", errors='coerce')
print(len(profundidad))

821


In [12]:
t0 = tiempo_fixed[0]  # o tiempo_dt[0] si es lista

# Calculamos el tiempo relativo en segundos
tiempo_relativo = (tiempo_fixed - t0).total_seconds()#/100

In [13]:
max_len = max(len(tiempos_5s), len(tiempo), len(tiempo_sonda))

In [14]:
# Suponiendo que tienes listas o arrays con estos datos
df_nuevo = pd.DataFrame({
    'Tiempo_rasp': pd.Series(tiempo_fixed).reindex(range(max_len), fill_value=-9999),
    'Tiempo_Relativo_rasp': pd.Series(tiempo_relativo).reindex(range(max_len), fill_value=-9999),
    'Profundidad_rasp': pd.Series(profundidad).reindex(range(max_len), fill_value=-9999),
    # 'Temperatura': pd.Series(temperatura).reindex(range(max_len), fill_value=-9999),
    # 'pH': pd.Series(ph).reindex(range(max_len), fill_value=-9999),
    # 'DO_SAT': pd.Series(do_sat).reindex(range(max_len), fill_value=-9999),
    # 'DO': pd.Series(do).reindex(range(max_len), fill_value=-9999),
    # 'Blue': pd.Series(blue).reindex(range(max_len), fill_value=-9999),
    # 'Chl': pd.Series(chl).reindex(range(max_len), fill_value=-9999),
    'Tiempo_sonda': pd.Series(date_sonda).reindex(range(max_len), fill_value=-9999),
    'Tiempo_Relativo_sonda': pd.Series(tiempo_sonda).reindex(range(max_len), fill_value=-9999),
    'Profundidad_sonda': pd.Series(profundidad_sonda).reindex(range(max_len), fill_value=-9999),
    'Tiempo_paparazzi': pd.Series(result_local_time).reindex(range(max_len), fill_value=-9999),
    'Tiempo_Relativo_paparazzi': pd.Series(tiempos_5s).reindex(range(max_len), fill_value=-9999),
    'Profundidad_paparazzi': pd.Series(profundidades_pprz).reindex(range(max_len), fill_value=-9999),
    'Static_control': pd.Series(static_control).reindex(range(max_len), fill_value=-9999),
    'Mode': pd.Series(mode).reindex(range(max_len), fill_value=-9999),
})



# Guardar en CSV
df_nuevo.to_csv(ruta_salida, na_rep=0, index=False)


In [15]:
df_prueba = pd.DataFrame({
    'Profundidad': profundidad
    
})

df_prueba.to_csv("Prueba.csv", na_rep=0, index=False)